In [1]:
from datasets import load_dataset
from tqdm import tqdm
from transformers import NllbTokenizer
from wtpsplit import SaT

from lcm_explo.adapters.embedding_repository.file_system._class import FileSystemEmbeddingRepository
from lcm_explo.domain.models.documents import DocumentEmbeddings
from lcm_explo.m2m_100 import M2M100EncoderModel

In [3]:
import numpy as np
import torch


def split_long_text(tokenizer, splitter, text: str, max_length: int = 1024) -> list[str]:
    tokens = tokenizer(text, return_tensors="pt")
    if tokens["input_ids"].shape[-1] <= max_length:
        return [text]
    splits = splitter.split(text)
    splits = [split for split in splits if split.replace(" ", "") != "" and len(split) > 50]
    return splits


def encode_text_sonar(text, device, sonar_tokenizer, sonar_model, splitter) -> np.ndarray:
    """Encode texts using SONAR model."""
    embeddings = []

    with torch.no_grad():
        chunks = split_long_text(sonar_tokenizer, splitter, text)
        for chunk in chunks:
            inputs = sonar_tokenizer(chunk, return_tensors="pt", padding=True, truncation=True).to(device)
            outputs = sonar_model(**inputs)
            embeddings.append(outputs.last_hidden_state.mean(dim=1))
    return torch.vstack(embeddings)

In [12]:
file_system_embedding = FileSystemEmbeddingRepository(base_path="./data")

wikipedia_dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
)

sonar_encoder = M2M100EncoderModel.from_pretrained("cointegrated/SONAR_200_text_encoder_hf").to("mps")
sonar_tokenizer = NllbTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M", src_lang="eng_Latn", tgt_lang="eng_Latn"
)

splitter = SaT("sat-3l")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [15]:
dataset_iter = iter(wikipedia_dataset)

for article in tqdm([next(dataset_iter) for _ in range(2000)]):
    encoded_article = encode_text_sonar(
        article["text"], device="mps", sonar_tokenizer=sonar_tokenizer, sonar_model=sonar_encoder, splitter=splitter
    )
    document = DocumentEmbeddings(document_id=article["title"], embeddings=encoded_article)

    file_system_embedding.save_document_embeddings(document)

Token indices sequence length is longer than the specified maximum sequence length for this model (10481 > 1024). Running this sequence through the model will result in indexing errors
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [2:35:58<00:00,  4.68s/it]
